<a href="https://colab.research.google.com/github/charansai12322/travel_buddy_agent/blob/main/travel_buddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U langchain langchain-google-genai langchain-tavily google-search-results

import os
import json
from google.colab import userdata

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from serpapi import GoogleSearch

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 676.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [4]:
def get_required_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise ValueError(
            f"Missing Colab Secret: {name}. "
            "Open the key icon in Colab, add the secret, and enable notebook access."
        )
    return value


GOOGLE_API_KEY = get_required_secret("sec1")
TAVILY_API_KEY = get_required_secret("TAVILY_API_KEY")
SERPAPI_API_KEY = get_required_secret("SERPAPI_API_KEY")

# Tavily reads this environment variable automatically.
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [5]:
travel_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
)

In [6]:
search_tool = TavilySearch(
    name="search_tool",
    max_results=5,
    search_depth="advanced",
    description=(
        "Search for destination attractions, culture, food, local transport, "
        "travel tips, and other travel-planning information."
    ),
)

In [7]:
@tool("serpapi_flights_tool")
def serpapi_flights_tool(origin: str, destination: str, date: str) -> str:
    """
    Search Google Flights through SerpAPI.

    Args:
        origin: Departure airport or city code, for example "HYD".
        destination: Arrival airport or city code, for example "GOI".
        date: One-way departure date in YYYY-MM-DD format.
    """
    params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": date,
        "type": 2,          # one-way flight search
        "currency": "INR",
        "hl": "en",
        "api_key": SERPAPI_API_KEY,
    }

    try:
        results = GoogleSearch(params).get_dict()

        if results.get("error"):
            return f"SerpAPI returned an error: {results['error']}"

        # Keep the response compact enough for the agent while preserving results.
        useful_results = {
            "best_flights": results.get("best_flights", [])[:5],
            "other_flights": results.get("other_flights", [])[:5],
            "price_insights": results.get("price_insights", {}),
            "search_metadata": results.get("search_metadata", {}),
        }

        return json.dumps(useful_results, indent=2, default=str)

    except Exception as error:
        return f"Flight search failed: {type(error).__name__}: {error}"

In [8]:
SYSTEM_PROMPT = """
You are a TravelBuddy assistant that helps travelers plan trips.

You have access to these tools:
- search_tool: Use it to search for attractions, culture, food, and travel tips.
- serpapi_flights_tool: Use it to search for flights using the Google Flights API.
  It requires origin, destination, and date.

Help the traveler by researching destinations and finding flights.
Present results in a clear, readable format with clear sections.
Do not use too much formatting.

When discussing flight prices, availability, or schedules, state that they can
change and should be verified before booking.
"""

In [9]:
travel_buddy_agent = create_agent(
    model=travel_model,
    tools=[search_tool, serpapi_flights_tool],
    system_prompt=SYSTEM_PROMPT,
)

In [10]:
query = """
I want to visit Goa from Hyderabad on 2026-12-15.
Show me top attractions and flight options.
"""

response = travel_buddy_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": query}
        ]
    }
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here's a plan for your trip to Goa:\n\n### Top Attractions in Goa\n\nGoa is famous for its beautiful beaches, rich history, and vibrant culture. Here are some of the top attractions you might want to visit:\n\n*   **Beaches:** Goa boasts numerous stunning white-sand beaches. Popular ones include Baga Beach, Anjuna Beach, Candolim Beach, Palolem Beach, Morjim Beach, and Arambol Beach. Many offer water sports and a lively atmosphere.\n*   **Basilica of Bom Jesus:** A UNESCO World Heritage site, this basilica is a significant landmark and houses the mortal remains of St. Francis Xavier.\n*   **Fontainhas:** Explore Panaji's charming Latin Quarter, known for its colorful Portuguese-style houses and narrow streets.\n*   **Dudhsagar Falls:** A spectacular four-tiered waterfall located on the Mandovi River, offering breathtaking views.\n*   **Se Cathedral Church:** One of the largest churches in Asia, showcasing impressive Portuguese-Gothic architecture.\n*   **Anju